# LeetCode 413 — Arithmetic Slices bằng Python

Link đề: https://leetcode.com/problems/arithmetic-slices/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def numberOfArithmeticSlices(self, nums: List[int]) -> int:
        ...
```

Ghi chú: phần phát biểu đề được dịch sang tiếng Việt từ đề gốc; mọi đáp án trong notebook được chạy lại và đối chiếu với một oracle vét cạn độc lập (kiểm tra từng đoạn con bằng định nghĩa).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **mảng con liên tiếp** (subarray) của `nums` có **ít nhất 3 phần tử** và có **hiệu hai phần tử liền kề không đổi** (cấp số cộng).
2. Nắm công cụ Python: cắt lát `nums[i:j+1]`, `zip(nums, nums[1:])`, `itertools.groupby`, `all(...)` với generator, danh sách hiệu `diff`, và phép chia `//` cho công thức tổng.
3. Đi từ cách **vét cạn `O(n^3)`** → **mở rộng đoạn con `O(n^2)`** → **DP theo "kết thúc tại `i`" `O(n)`** → **chỉ giữ một biến đếm `O(1)` space** → **công thức theo từng "dải" hiệu bằng nhau**.
4. Hiểu mấu chốt: nếu `nums[i] - nums[i-1] == nums[i-1] - nums[i-2]` thì **mọi** cấp số cộng kết thúc ở `i-1` đều kéo dài thêm được, cộng thêm **một** cấp số mới độ dài 3 ⇒ `dp[i] = dp[i-1] + 1`.
5. Tránh các lỗi kinh điển: quên reset bộ đếm khi hiệu đổi, quên cộng dồn `dp` vào đáp án, nhầm "subarray" (liên tiếp) với "subsequence" (không liên tiếp), tính sai công thức dải `(L-1)(L-2)/2`, nhầm điều kiện "ít nhất 3 phần tử".

## 2. Đề bài & ví dụ

Một mảng số nguyên được gọi là **cấp số cộng** (arithmetic) nếu nó có **ít nhất ba phần tử** và hiệu giữa hai phần tử liền kề bất kỳ **bằng nhau**.

- Ví dụ `[1,3,5,7,9]`, `[7,7,7,7]`, `[3,-1,-5,-9]` đều là cấp số cộng.

Cho mảng `nums`, trả về **số lượng mảng con** (subarray, tức đoạn **liên tiếp**) của `nums` là cấp số cộng.

**Ví dụ 1:** `nums = [1,2,3,4]` → `3`. Có 3 đoạn: `[1,2,3]`, `[2,3,4]` và `[1,2,3,4]`.

**Ví dụ 2:** `nums = [1]` → `0`.

**Ràng buộc:**

- `1 <= nums.length <= 5000`
- `-1000 <= nums[i] <= 1000`

In [1]:
from itertools import groupby
import random
import time
from typing import List

examples = [
    ([1, 2, 3, 4], 3),
    ([1], 0),
]


def is_arithmetic(a):
    """Định nghĩa trực tiếp: ít nhất 3 phần tử và các hiệu liền kề bằng nhau."""
    return len(a) >= 3 and all(a[i + 1] - a[i] == a[1] - a[0] for i in range(len(a) - 1))


for nums, expected in examples:
    print(nums, "->", expected)
print(is_arithmetic([1, 3, 5, 7, 9]), is_arithmetic([7, 7, 7, 7]), is_arithmetic([3, -1, -5, -9]), is_arithmetic([1, 2]))

[1, 2, 3, 4] -> 3
[1] -> 0
True True True False


### 2.1. Đọc kỹ đề

- **Subarray** = đoạn **liên tiếp** `nums[i..j]`. Không phải subsequence (chọn rời rạc). Hai đoạn cùng giá trị nhưng khác vị trí được tính là **hai** đoạn khác nhau.
- Mảng có đúng `n` phần tử thì có `n * (n + 1) / 2` đoạn con; ta chỉ đếm những đoạn có `>= 3` phần tử **và** hiệu không đổi.
- `[7,7,7,7]` là cấp số cộng với hiệu `0`. Mảng toàn số bằng nhau cho đáp án lớn nhất.
- Đáp án có thể lớn: `n = 5000` toàn số bằng nhau cho `(n-1)(n-2)/2 = 12,492,501` — vẫn vừa số nguyên 32-bit.

## 3. Kiến thức nền

### 3.1. Cắt lát (slicing) lấy đoạn con

`nums[i:j]` lấy các phần tử từ chỉ số `i` đến `j - 1` (**không gồm** `j`). Để lấy đoạn `nums[i..j]` gồm cả `j` ta viết `nums[i:j + 1]`.

In [2]:
nums = [1, 2, 3, 4]
print(nums[0:3], nums[1:4], nums[0:4], nums[2:3])
i, j = 1, 3
print("đoạn nums[1..3] =", nums[i:j + 1], "độ dài", j - i + 1)

[1, 2, 3] [2, 3, 4] [1, 2, 3, 4] [3]
đoạn nums[1..3] = [2, 3, 4] độ dài 3


### 3.2. Danh sách hiệu `diff`

Một dãy là cấp số cộng khi **các hiệu liền kề bằng nhau**. Tạo danh sách hiệu `diff[i] = nums[i+1] - nums[i]` bằng `zip(nums, nums[1:])`: `zip` ghép phần tử thứ `i` của `nums` với phần tử thứ `i` của `nums[1:]` (tức `nums[i+1]`).

In [3]:
nums = [1, 2, 3, 4, 7, 10, 13, 13]
diff = [b - a for a, b in zip(nums, nums[1:])]
print(diff)                                # [1, 1, 1, 3, 3, 3, 0]
print(len(nums), len(diff))                # diff ngắn hơn 1

[1, 1, 1, 3, 3, 3, 0]
8 7


### 3.3. `all(...)` với generator

`all(iterable)` trả `True` nếu mọi phần tử đều thật. Viết gọn "mọi hiệu bằng nhau": `all(x == d for x in diffs)`. Với dãy rỗng `all([])` là `True`.

In [4]:
d = 2
print(all(x == d for x in [2, 2, 2]), all(x == d for x in [2, 3, 2]), all([]))

True False True


### 3.4. `itertools.groupby`

`groupby(iterable)` gom các phần tử **liền kề bằng nhau** thành nhóm: mỗi nhóm cho `(giá trị, iterator các phần tử)`. Ta dùng nó để tìm các "dải" hiệu bằng nhau.

In [5]:
diff = [1, 1, 1, 3, 3, 3, 0]
for value, grp in groupby(diff):
    print("hiệu", value, "lặp", len(list(grp)), "lần")

hiệu 1 lặp 3 lần
hiệu 3 lặp 3 lần
hiệu 0 lặp 1 lần


### 3.5. Tổng `1 + 2 + ... + m`

`1 + 2 + ... + m = m * (m + 1) // 2`. Ta sẽ gặp tổng này khi đếm số cấp số cộng trong một dải. Kiểm tra nhanh:

In [6]:
for m in range(0, 6):
    assert sum(range(1, m + 1)) == m * (m + 1) // 2
print([m * (m + 1) // 2 for m in range(0, 7)])

[0, 1, 3, 6, 10, 15, 21]


## 4. Ý tưởng nền tảng: tính tay

Xét `nums = [1, 2, 3, 4]`. Hiệu liền kề: `[1, 1, 1]`.

Đếm các đoạn con có độ dài `>= 3`:

- Độ dài 3: `[1,2,3]` (hiệu 1,1 ✓), `[2,3,4]` (hiệu 1,1 ✓) ⇒ 2 đoạn.
- Độ dài 4: `[1,2,3,4]` (hiệu 1,1,1 ✓) ⇒ 1 đoạn.

Tổng `3` ✓.

Bây giờ xét `[1, 2, 3, 4, 6]`: hiệu `[1, 1, 1, 2]`. Đoạn `[3,4,6]` có hiệu `1, 2` ✗; `[2,3,4,6]` ✗; `[1,2,3,4,6]` ✗. Vậy chỉ có 3 đoạn, giống như `[1,2,3,4]`: phần tử `6` làm **đổi hiệu** nên không tạo thêm đoạn nào kết thúc tại nó.

**Nhận xét:** một cấp số cộng kết thúc tại vị trí `i` (với `i >= 2`) chỉ có thể tồn tại khi hiệu `nums[i]-nums[i-1]` bằng hiệu cũ `nums[i-1]-nums[i-2]`.

## 5. Cách 1 — Brute Force: thử mọi đoạn con

**Ý tưởng:** với mọi cặp `(i, j)` với `j - i + 1 >= 3`, cắt `nums[i:j+1]` và kiểm tra bằng `is_arithmetic`. Đếm số đoạn thỏa.

In [7]:
def count_brute(nums):
    n = len(nums)
    cnt = 0
    for i in range(n):
        for j in range(i + 2, n):                    # độ dài >= 3
            if is_arithmetic(nums[i:j + 1]):
                cnt += 1
    return cnt


for nums, expected in examples:
    got = count_brute(nums)
    print(nums, "->", got)
    assert got == expected

[1, 2, 3, 4] -> 3
[1] -> 0


**Độ phức tạp:** `O(n^2)` cặp `(i, j)`, mỗi lần cắt và kiểm tra tốn `O(n)` ⇒ Time `O(n^3)`, Space `O(n)` (đoạn cắt ra).

**Nút thắt:** với `n = 5000` cỡ `2 * 10^10` thao tác. Hãy đo để thấy nó tăng nhanh thế nào.

In [8]:
for n in [50, 100, 200]:
    arr = list(range(n))
    t = time.perf_counter()
    r = count_brute(arr)
    dt = time.perf_counter() - t
    print(f"n={n:>3}: đáp án {r:>6}, mất {dt * 1000:8.1f} ms")

n= 50: đáp án   1176, mất      1.5 ms
n=100: đáp án   4851, mất     11.8 ms
n=200: đáp án  19701, mất     83.2 ms


## 6. Ý tưởng thuật toán / bước đột phá

Brute force kiểm tra lại từ đầu những đoạn chồng nhau. Hai quan sát:

1. **Mở rộng dần:** cố định đầu trái `i`. Nếu `[i..j]` là cấp số cộng với hiệu `d` thì `[i..j+1]` là cấp số cộng khi và chỉ khi `nums[j+1] - nums[j] == d`. Khi hiệu đổi, **mọi** đoạn dài hơn bắt đầu tại `i` đều hỏng ⇒ dừng. Việc này giúp bỏ nhân tố `O(n)` của việc kiểm tra: `O(n^2)`.
2. **Đếm theo điểm kết thúc:** gọi `dp[i]` = số cấp số cộng (độ dài `>= 3`) **kết thúc đúng tại `i`**. Nếu `nums[i] - nums[i-1] == nums[i-1] - nums[i-2]`:
   - mọi cấp số kết thúc tại `i-1` (có `dp[i-1]` cái) kéo dài thêm `nums[i]` vẫn là cấp số;
   - thêm **một** cấp số mới: `[nums[i-2], nums[i-1], nums[i]]`.

   Vậy `dp[i] = dp[i-1] + 1`. Ngược lại `dp[i] = 0`. Đáp án là `sum(dp)`.

Ví dụ `[1,2,3,4]`: `dp = [0, 0, 1, 2]` (tại `i = 2`: `[1,2,3]`; tại `i = 3`: `[2,3,4]` và `[1,2,3,4]`), tổng `3` ✓.

In [9]:
nums = [1, 2, 3, 4, 6, 8, 10, 10]
dp = [0] * len(nums)
for i in range(2, len(nums)):
    if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
        dp[i] = dp[i - 1] + 1
print("nums =", nums)
print("dp   =", dp, "-> tổng", sum(dp), "| brute =", count_brute(nums))
assert sum(dp) == count_brute(nums)

nums = [1, 2, 3, 4, 6, 8, 10, 10]
dp   = [0, 0, 1, 2, 0, 1, 2, 0] -> tổng 6 | brute = 6


## 7. Các cách tiếp theo

### Cách 2 — Mở rộng đoạn con từ mỗi đầu trái `O(n^2)`

**Giải thích từng đoạn:**

1. Với mỗi `i` từ `0` đến `n - 3`: lấy hiệu `d = nums[i+1] - nums[i]`.
2. Cho `j` chạy từ `i + 2` đến hết: nếu `nums[j] - nums[j-1] == d` thì `[i..j]` là cấp số cộng ⇒ `cnt += 1`; ngược lại `break` (mọi đoạn dài hơn cũng hỏng).

In [10]:
def count_extend(nums):
    n = len(nums)
    cnt = 0
    for i in range(n - 2):
        d = nums[i + 1] - nums[i]
        for j in range(i + 2, n):
            if nums[j] - nums[j - 1] == d:
                cnt += 1
            else:
                break
    return cnt


for nums, expected in examples:
    got = count_extend(nums)
    print(nums, "->", got)
    assert got == expected

[1, 2, 3, 4] -> 3
[1] -> 0


In [11]:
# Code nộp LeetCode — Cách 2 (O(n^2) time, O(1) space)
class SolutionExtend:
    def numberOfArithmeticSlices(self, nums: List[int]) -> int:
        n = len(nums)
        cnt = 0
        for i in range(n - 2):
            d = nums[i + 1] - nums[i]
            for j in range(i + 2, n):
                if nums[j] - nums[j - 1] == d:
                    cnt += 1
                else:
                    break
        return cnt


print([SolutionExtend().numberOfArithmeticSlices(nums) for nums, _ in examples])

[3, 0]


**Độ phức tạp Cách 2:** hai vòng lặp lồng nhau, tệ nhất (mảng toàn số bằng nhau) `O(n^2)`; Space `O(1)`. Với `n = 5000` khoảng `1.25 * 10^7` bước — vẫn đủ nhanh trên LeetCode, nhưng ta còn tốt hơn được.

### Cách 3 — DP `dp[i]` = số cấp số cộng kết thúc tại `i`

**Giải thích từng đoạn:**

1. `dp = [0] * n`.
2. Với `i` từ `2` đến `n - 1`: nếu `nums[i] - nums[i-1] == nums[i-1] - nums[i-2]` thì `dp[i] = dp[i-1] + 1`.
3. Đáp án `sum(dp)`.

In [12]:
def count_dp(nums):
    n = len(nums)
    dp = [0] * n
    for i in range(2, n):
        if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
            dp[i] = dp[i - 1] + 1
    return sum(dp)


for nums, expected in examples:
    got = count_dp(nums)
    print(nums, "->", got)
    assert got == expected

[1, 2, 3, 4] -> 3
[1] -> 0


In [13]:
# Code nộp LeetCode — Cách 3 (O(n) time, O(n) space)
class SolutionDP:
    def numberOfArithmeticSlices(self, nums: List[int]) -> int:
        n = len(nums)
        dp = [0] * n
        for i in range(2, n):
            if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
                dp[i] = dp[i - 1] + 1
        return sum(dp)


print([SolutionDP().numberOfArithmeticSlices(nums) for nums, _ in examples])

[3, 0]


**Độ phức tạp Cách 3:** một vòng lặp ⇒ Time `O(n)`, Space `O(n)`.

### Cách 4 — Một biến đếm `cnt` (tối ưu bộ nhớ, chốt)

`dp[i]` chỉ phụ thuộc `dp[i-1]` nên ta thay mảng bằng **một biến** `cur` (chính là `dp[i-1]`) và cộng dồn vào `ans`.

**Giải thích từng đoạn:**

1. `ans = 0`, `cur = 0` (số cấp số kết thúc tại vị trí trước).
2. Với `i` từ `2` đến `n - 1`: nếu hiệu giữ nguyên thì `cur += 1`, ngược lại `cur = 0`.
3. `ans += cur`.

In [14]:
def count_counter(nums):
    ans = cur = 0
    for i in range(2, len(nums)):
        if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
            cur += 1
        else:
            cur = 0
        ans += cur
    return ans


for nums, expected in examples:
    got = count_counter(nums)
    print(nums, "->", got)
    assert got == expected

[1, 2, 3, 4] -> 3
[1] -> 0


In [15]:
# Code nộp LeetCode — Cách 4 (O(n) time, O(1) space)
class SolutionCounter:
    def numberOfArithmeticSlices(self, nums: List[int]) -> int:
        ans = cur = 0
        for i in range(2, len(nums)):
            if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
                cur += 1
            else:
                cur = 0
            ans += cur
        return ans


print([SolutionCounter().numberOfArithmeticSlices(nums) for nums, _ in examples])

[3, 0]


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(1)`.

### Cách 5 — Công thức theo từng "dải" hiệu bằng nhau

**Góc nhìn khác:** chia danh sách hiệu `diff` thành các **dải** liên tiếp có cùng giá trị. Một dải gồm `m` hiệu bằng nhau ứng với một đoạn `m + 1` phần tử của `nums` là cấp số cộng cực đại. Trong đoạn `L = m + 1` phần tử đó, số đoạn con có độ dài `>= 3` là:

- độ dài 3: `L - 2` đoạn; độ dài 4: `L - 3` đoạn; ...; độ dài `L`: `1` đoạn.
- Tổng `1 + 2 + ... + (L - 2) = (L - 2)(L - 1) / 2 = (m - 1) * m / 2`.

Các dải **chỉ chia sẻ** ranh giới nhưng một đoạn con có độ dài `>= 3` chứa ít nhất 2 hiệu mà phải bằng nhau nên nó nằm hoàn toàn trong **một** dải; do đó cộng các dải là đúng, không đếm trùng.

**Giải thích từng đoạn:**

1. Tính `diff = [b - a for a, b in zip(nums, nums[1:])]`.
2. `groupby(diff)` cho từng dải; `m = số phần tử của dải`.
3. `ans += m * (m - 1) // 2`.

In [16]:
def count_runs(nums):
    diff = [b - a for a, b in zip(nums, nums[1:])]
    ans = 0
    for _, grp in groupby(diff):
        m = len(list(grp))                          # số hiệu liên tiếp bằng nhau
        ans += m * (m - 1) // 2
    return ans


for nums, expected in examples:
    got = count_runs(nums)
    print(nums, "->", got)
    assert got == expected
print("[1,2,3,4,6,8,10,10] ->", count_runs([1, 2, 3, 4, 6, 8, 10, 10]))

[1, 2, 3, 4] -> 3
[1] -> 0
[1,2,3,4,6,8,10,10] -> 6


In [17]:
# Code nộp LeetCode — Cách 5 (O(n) time, O(n) space)
from itertools import groupby


class SolutionRuns:
    def numberOfArithmeticSlices(self, nums: List[int]) -> int:
        diff = [b - a for a, b in zip(nums, nums[1:])]
        ans = 0
        for _, grp in groupby(diff):
            m = sum(1 for _ in grp)
            ans += m * (m - 1) // 2
        return ans


print([SolutionRuns().numberOfArithmeticSlices(nums) for nums, _ in examples])

[3, 0]


**Độ phức tạp Cách 5:** Time `O(n)`, Space `O(n)` cho `diff` (có thể làm `O(1)` nếu duyệt thẳng, nhưng ở đây ưu tiên rõ ràng).

## 8. TRACE từng bước

Trace Cách 4 với `nums = [1, 2, 3, 4, 6, 8, 10, 10]`:

| `i` | `nums[i-2], nums[i-1], nums[i]` | hiệu cũ | hiệu mới | bằng? | `cur` | `ans` |
|---|---|---|---|---|---|---|
| 2 | 1, 2, 3 | 1 | 1 | có | 1 | 1 |
| 3 | 2, 3, 4 | 1 | 1 | có | 2 | 3 |
| 4 | 3, 4, 6 | 1 | 2 | không | 0 | 3 |
| 5 | 4, 6, 8 | 2 | 2 | có | 1 | 4 |
| 6 | 6, 8, 10 | 2 | 2 | có | 2 | 6 |
| 7 | 8, 10, 10 | 2 | 0 | không | 0 | 6 |

Đáp án `6`. Hàm `trace_counter` in đúng bảng này, `trace_runs` in các dải của Cách 5.

In [18]:
def trace_counter(nums):
    print("nums =", nums)
    ans = cur = 0
    print(f"{'i':>2} | {'bộ ba':>12} | hiệu cũ | hiệu mới | bằng? | cur | ans")
    for i in range(2, len(nums)):
        old = nums[i - 1] - nums[i - 2]
        new = nums[i] - nums[i - 1]
        if new == old:
            cur += 1
        else:
            cur = 0
        ans += cur
        triple = f"{nums[i - 2]},{nums[i - 1]},{nums[i]}"
        print(f"{i:>2} | {triple:>12} | {old:>7} | {new:>8} | {'có' if new == old else 'không':>5} | {cur:>3} | {ans:>3}")
    print("đáp án:", ans)
    return ans


trace_counter([1, 2, 3, 4, 6, 8, 10, 10])

nums = [1, 2, 3, 4, 6, 8, 10, 10]
 i |        bộ ba | hiệu cũ | hiệu mới | bằng? | cur | ans
 2 |        1,2,3 |       1 |        1 |    có |   1 |   1
 3 |        2,3,4 |       1 |        1 |    có |   2 |   3
 4 |        3,4,6 |       1 |        2 | không |   0 |   3
 5 |        4,6,8 |       2 |        2 |    có |   1 |   4
 6 |       6,8,10 |       2 |        2 |    có |   2 |   6
 7 |      8,10,10 |       2 |        0 | không |   0 |   6
đáp án: 6


6

In [19]:
def trace_runs(nums):
    diff = [b - a for a, b in zip(nums, nums[1:])]
    print("nums =", nums)
    print("diff =", diff)
    ans = 0
    for value, grp in groupby(diff):
        m = len(list(grp))
        add = m * (m - 1) // 2
        ans += add
        print(f"dải hiệu {value:>2} dài m={m} (đoạn {m + 1} phần tử) -> m*(m-1)//2 = {add}")
    print("đáp án:", ans)
    return ans


trace_runs([1, 2, 3, 4, 6, 8, 10, 10])
print()
trace_runs([7, 7, 7, 7, 7])

nums = [1, 2, 3, 4, 6, 8, 10, 10]
diff = [1, 1, 1, 2, 2, 2, 0]
dải hiệu  1 dài m=3 (đoạn 4 phần tử) -> m*(m-1)//2 = 3
dải hiệu  2 dài m=3 (đoạn 4 phần tử) -> m*(m-1)//2 = 3
dải hiệu  0 dài m=1 (đoạn 2 phần tử) -> m*(m-1)//2 = 0
đáp án: 6

nums = [7, 7, 7, 7, 7]
diff = [0, 0, 0, 0]
dải hiệu  0 dài m=4 (đoạn 5 phần tử) -> m*(m-1)//2 = 6
đáp án: 6


6

## 9. Code nộp LeetCode

Lời giải chính là **Cách 4** (một biến đếm): `O(n)` thời gian, `O(1)` bộ nhớ, rất ngắn.

In [20]:
class Solution:
    def numberOfArithmeticSlices(self, nums: List[int]) -> int:
        ans = 0                                           # tổng số đoạn cấp số cộng
        cur = 0                                           # số cấp số cộng kết thúc tại vị trí i - 1
        for i in range(2, len(nums)):
            if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
                cur += 1                                  # kéo dài mọi đoạn cũ + 1 đoạn mới độ dài 3
            else:
                cur = 0                                   # hiệu đổi: không còn đoạn nào kết thúc ở i
            ans += cur
        return ans


for nums, expected in examples:
    got = Solution().numberOfArithmeticSlices(nums)
    print(nums, "->", got)
    assert got == expected

[1, 2, 3, 4] -> 3
[1] -> 0


## 10. Vì sao đúng

1. **Phân loại theo điểm kết thúc:** mỗi đoạn con có độ dài `>= 3` kết thúc tại đúng một chỉ số `i >= 2`. Đếm theo `i` rồi cộng lại không bỏ sót, không trùng.
2. **Điều kiện cần:** một cấp số cộng độ dài `>= 3` kết thúc tại `i` có hai hiệu cuối `nums[i-1]-nums[i-2]` và `nums[i]-nums[i-1]` bằng nhau. Nếu chúng khác nhau thì `dp[i] = 0`.
3. **Khi hiệu bằng nhau:** các đoạn kết thúc tại `i` gồm (a) đoạn `[i-2, i-1, i]` (độ dài 3) và (b) các đoạn độ dài `>= 4`, mà bỏ phần tử cuối đi thì là đoạn kết thúc tại `i-1` với **cùng hiệu** — và ngược lại mọi cấp số kết thúc tại `i-1` (hiệu cuối đúng bằng hiệu hiện tại, vì `nums[i-1]-nums[i-2]` là hiệu cuối của chúng) kéo dài được. Do đó `dp[i] = dp[i-1] + 1`.
4. **`cur` chính là `dp[i-1]`:** chỉ cần giá trị liền trước nên một biến là đủ.
5. **Công thức dải (Cách 5):** trong một dải dài `m` hiệu bằng nhau có `dp = 1, 2, ..., m - 1` ⇒ tổng `(m-1) * m / 2`, trùng với Cách 4 (ta sẽ kiểm bằng `assert` ở mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn mọi đoạn | `O(n^3)` | `O(n)` |
| 2. Mở rộng từ đầu trái | `O(n^2)` | `O(1)` |
| 3. DP mảng | `O(n)` | `O(n)` |
| 4. Một biến đếm | **`O(n)`** | **`O(1)`** |
| 5. Công thức theo dải | `O(n)` | `O(n)` |

Với `n = 5000`, Cách 4 chỉ vài nghìn phép so sánh.

## 12. Bộ test

Oracle độc lập: `count_brute` dùng **định nghĩa** `is_arithmetic` trên từng đoạn con (mục 5). Ta đối chiếu năm cách với nó trên input nhỏ (vét cạn mọi mảng trên bảng chữ cái nhỏ), rồi ngẫu nhiên có seed, rồi input lớn đối chiếu giữa các cách nhanh.

In [21]:
SOLUTIONS = [Solution(), SolutionExtend(), SolutionDP(), SolutionCounter(), SolutionRuns()]

# 12.1 Ví dụ của đề + edge case
cases = [
    ([1, 2, 3, 4], 3),
    ([1], 0),
    ([1, 2], 0),
    ([1, 2, 3], 1),
    ([1, 2, 4], 0),
    ([7, 7, 7, 7], 3),
    ([7, 7, 7, 7, 7], 6),
    ([3, -1, -5, -9], 3),
    ([1, 3, 5, 7, 9], 6),
    ([1, 2, 3, 8, 9, 10], 2),
    ([1, 2, 3, 4, 6, 8, 10, 10], 6),
    ([1, 3, 2, 4, 3, 5], 0),                       # tăng giảm xen kẽ: không có đoạn nào
    ([-1000, 0, 1000], 1),                         # biên giá trị
    ([1000, 1000, 1000], 1),
    ([1, 2, 3, 3, 3, 3], 1 + 3),                   # dải 1,1 (1 đoạn) + dải 0,0,0 (3 đoạn)
]
for nums, expected in cases:
    assert count_brute(nums) == expected, (nums, count_brute(nums), expected)
    for sol in SOLUTIONS:
        got = sol.numberOfArithmeticSlices(nums)
        assert got == expected, (type(sol).__name__, nums, got, expected)
print("OK:", len(cases), "test x", len(SOLUTIONS), "lớp (đã đối chiếu với vét cạn)")

OK: 15 test x 5 lớp (đã đối chiếu với vét cạn)


In [22]:
# 12.2 Vét cạn mọi mảng độ dài <= 8 trên bảng chữ cái {0,1,2} và {0,2,4}
from itertools import product

count = 0
for alphabet in [(0, 1, 2), (0, 2, 4)]:
    for n in range(1, 9):
        for arr in product(alphabet, repeat=n):
            nums = list(arr)
            expected = count_brute(nums)
            for sol in SOLUTIONS:
                assert sol.numberOfArithmeticSlices(nums) == expected, (type(sol).__name__, nums)
            count += 1
print("OK: đối chiếu", count, "mảng với vét cạn")

OK: đối chiếu 19680 mảng với vét cạn


In [23]:
# 12.3 Ngẫu nhiên có seed, đúng ràng buộc giá trị [-1000, 1000]; n <= 60 so với vét cạn
random.seed(413)
for _ in range(2000):
    n = random.randint(1, 60)
    mode = random.choice(["rand", "small", "arith"])
    if mode == "rand":
        nums = [random.randint(-1000, 1000) for _ in range(n)]
    elif mode == "small":
        nums = [random.randint(-2, 2) for _ in range(n)]
    else:                                           # ghép các dải cấp số cộng
        nums = []
        while len(nums) < n:
            a, d, L = random.randint(-100, 100), random.randint(-5, 5), random.randint(1, 8)
            nums += [a + d * t for t in range(L)]
        nums = nums[:n]
    expected = count_brute(nums)
    for sol in SOLUTIONS:
        assert sol.numberOfArithmeticSlices(nums) == expected, (type(sol).__name__, nums)
print("OK: 2000 mảng ngẫu nhiên n <= 60")

OK: 2000 mảng ngẫu nhiên n <= 60


In [24]:
# 12.4 Input lớn n = 5000: các cách nhanh thống nhất + công thức đóng cho mảng cấp số cộng toàn phần
random.seed(4131)
for _ in range(20):
    n = 5000
    nums = [random.randint(-3, 3) for _ in range(n)]
    results = {sol.numberOfArithmeticSlices(nums) for sol in SOLUTIONS}
    assert len(results) == 1, results
n = 5000
assert Solution().numberOfArithmeticSlices([5] * n) == (n - 1) * (n - 2) // 2 == 12_492_501
assert Solution().numberOfArithmeticSlices(list(range(-1000, 1001))) == (2001 - 1) * (2001 - 2) // 2
print("OK: n = 5000, toàn số bằng nhau ->", Solution().numberOfArithmeticSlices([5] * n))

OK: n = 5000, toàn số bằng nhau -> 12492501


In [25]:
# 12.5 Phản ví dụ: subsequence KHÔNG phải subarray
def count_subsequences_arith(nums):
    """Đếm dãy con (chọn rời rạc) có >= 3 phần tử, cấp số cộng: KHÔNG phải đề bài này."""
    from itertools import combinations
    cnt = 0
    for r in range(3, len(nums) + 1):
        for idx in combinations(range(len(nums)), r):
            vals = [nums[i] for i in idx]
            if is_arithmetic(vals):
                cnt += 1
    return cnt


nums = [1, 2, 3, 4, 5]
print("subarray:", Solution().numberOfArithmeticSlices(nums), "| subsequence:", count_subsequences_arith(nums))
assert Solution().numberOfArithmeticSlices(nums) == 6
assert count_subsequences_arith(nums) == 7                # có thêm [1,3,5]
# 12.6 Hiệu năng
t = time.perf_counter()
for _ in range(200):
    Solution().numberOfArithmeticSlices([5] * 5000)
dt = time.perf_counter() - t
assert dt < 5
print(f"200 lần n=5000 mất {dt:.2f} giây")

subarray: 6 | subsequence: 7
200 lần n=5000 mất 0.10 giây


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên reset bộ đếm khi hiệu thay đổi

In [26]:
def wrong_no_reset(nums):
    ans = cur = 0
    for i in range(2, len(nums)):
        if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
            cur += 1
        ans += cur                                       # SAI: thiếu nhánh else: cur = 0
    return ans


nums = [1, 2, 3, 4, 6, 8, 10, 10]
print("sai:", wrong_no_reset(nums), "| đúng:", Solution().numberOfArithmeticSlices(nums))
assert wrong_no_reset(nums) != Solution().numberOfArithmeticSlices(nums)

sai: 16 | đúng: 6


**Vì sao sai:** khi hiệu đổi, không còn đoạn nào kết thúc tại `i`, nhưng `cur` cũ vẫn được cộng lại vào `ans` ở các bước sau. **Cách sửa:** `else: cur = 0`.

### Lỗi 2: quên cộng dồn mỗi bước (chỉ lấy `cur` cuối)

In [27]:
def wrong_last_only(nums):
    cur = 0
    for i in range(2, len(nums)):
        if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
            cur += 1
        else:
            cur = 0
    return cur                                           # SAI: trả về cur cuối thay vì tổng


nums = [1, 2, 3, 4, 6, 8, 10, 10]
print("sai:", wrong_last_only(nums), "| đúng:", Solution().numberOfArithmeticSlices(nums))
assert wrong_last_only(nums) != Solution().numberOfArithmeticSlices(nums)

sai: 0 | đúng: 6


**Vì sao sai:** `cur` chỉ đếm đoạn **kết thúc tại vị trí hiện tại**. Đáp án là tổng của chúng qua **mọi** vị trí. **Cách sửa:** `ans += cur` ở mỗi bước.

### Lỗi 3: dùng `+= 1` mỗi lần hiệu bằng nhau (bỏ quên đoạn dài)

In [28]:
def wrong_plus_one(nums):
    ans = 0
    for i in range(2, len(nums)):
        if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
            ans += 1                                     # SAI: chỉ đếm đoạn độ dài 3, bỏ các đoạn dài hơn
    return ans


print("sai:", wrong_plus_one([1, 2, 3, 4]), "| đúng:", Solution().numberOfArithmeticSlices([1, 2, 3, 4]))
assert wrong_plus_one([1, 2, 3, 4]) == 2 != 3

sai: 2 | đúng: 3


**Vì sao sai:** mỗi lần hiệu giữ nguyên, ta thêm **một** đoạn độ dài 3 **và** kéo dài mọi đoạn cũ (`dp[i-1]` đoạn). `+= 1` chỉ đếm đoạn mới. **Cách sửa:** `cur += 1; ans += cur`.

### Lỗi 4: nhầm subarray với subsequence

Xem mục 12.5: `[1,2,3,4,5]` có 6 subarray cấp số cộng nhưng 7 subsequence (thêm `[1,3,5]`). Đề hỏi **liên tiếp**, nên `[1,3,5]` **không** được tính. **Cách sửa:** chỉ xét đoạn liên tiếp `nums[i..j]`.

### Lỗi 5: tính sai số đoạn trong dải (off-by-one của công thức)

In [29]:
def wrong_formula(nums):
    diff = [b - a for a, b in zip(nums, nums[1:])]
    ans = 0
    for _, grp in groupby(diff):
        m = len(list(grp))
        ans += m * (m + 1) // 2                          # SAI: phải là m * (m - 1) // 2
    return ans


print("sai:", wrong_formula([1, 2, 3, 4]), "| đúng:", Solution().numberOfArithmeticSlices([1, 2, 3, 4]))
print("sai với [1, 2]:", wrong_formula([1, 2]), "| đúng 0")
assert wrong_formula([1, 2]) == 1 != 0

sai: 6 | đúng: 3
sai với [1, 2]: 1 | đúng 0


**Vì sao sai:** dải `m` hiệu ứng với `m + 1` phần tử và `m - 1` là số đoạn độ dài 3 lớn nhất; tổng là `1 + ... + (m - 1) = m * (m - 1) / 2`. Dùng `m * (m + 1) / 2` đếm cả các đoạn có 2 phần tử. **Cách sửa:** `m * (m - 1) // 2`; hai phần tử (`m = 1`) cho `0`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu nhận dạng:** "đếm số đoạn con liên tiếp thỏa một tính chất mà tính chất đó **được duy trì khi mở rộng sang phải**" ⇒ **đếm theo điểm kết thúc** với biến `cur`. Cùng họ: *Number of Zero-Filled Subarrays* (2348), *Number of Substrings With Only 1s* (1513), *Longest Turbulent Subarray* (978), *Arithmetic Slices II - Subsequence* (446).
- **Công thức nhanh:** `cur` tăng 1 khi tiếp tục, về `0` khi đứt; `ans += cur`.
- **Dải hiệu bằng nhau** đếm bằng `m * (m - 1) // 2` hoặc bằng cộng dồn `cur`.
- **Subarray ≠ subsequence:** liên tiếp thì đếm theo cửa sổ/dp 1 chiều; rời rạc (446) cần dp với dictionary theo hiệu.
- Biên: `n < 3` ⇒ đáp án `0` (vòng `range(2, n)` rỗng tự nhiên).
- Kiểm chứng bằng vét cạn theo định nghĩa trên mảng ngắn.

## 15. Biến thể

1. **Độ dài cấp số cộng dài nhất** (liên tiếp): giữ `cur` là độ dài dải hiện tại, lấy `max`.
2. **Đếm cấp số cộng độ dài `>= L` bất kỳ:** với mỗi dải `m` hiệu, cộng `(m + 2 - len)` cho `len = L .. m + 1` (làm ở bài tập 5).
3. **Cấp số nhân** (liên tiếp): thay hiệu bằng tỉ số (cẩn thận chia cho `0`, dùng tích chéo `a*c == b*b`).
4. **Subsequence cấp số cộng** (LeetCode 446): cần dictionary `dp[i][d]`.
5. **Đoạn liên tiếp mà hiệu luân phiên dấu** (turbulent, LeetCode 978).

In [30]:
def longest_arith_run(nums):
    """Độ dài dài nhất của đoạn liên tiếp là cấp số cộng (>= 2 phần tử nếu n >= 2)."""
    n = len(nums)
    if n <= 2:
        return n
    best = cur = 2
    for i in range(2, n):
        if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
            cur += 1
        else:
            cur = 2
        best = max(best, cur)
    return best


print(longest_arith_run([1, 2, 3, 4, 6, 8, 10, 12, 14]), longest_arith_run([5]), longest_arith_run([1, 5]))


def count_geometric_slices(nums):
    """Biến thể: đoạn liên tiếp >= 3 phần tử là cấp số nhân (các số khác 0, kiểm tra bằng tích chéo a*c == b*b)."""
    ans = cur = 0
    for i in range(2, len(nums)):
        a, b, c = nums[i - 2], nums[i - 1], nums[i]
        if a != 0 and b != 0 and a * c == b * b:
            cur += 1
        else:
            cur = 0
        ans += cur
    return ans


print(count_geometric_slices([1, 2, 4, 8]), count_geometric_slices([2, 6, 18, 54, 55]))
assert count_geometric_slices([1, 2, 4, 8]) == 3

6 1 2
3 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn mọi đoạn | cắt và kiểm tra từng đoạn bằng định nghĩa | `O(n^3)` | `O(n)` | **oracle** kiểm chứng |
| 2. Mở rộng từ đầu trái | cố định đầu, kéo dài tới khi hiệu đổi | `O(n^2)` | `O(1)` | hiểu bản chất, `n` nhỏ |
| 3. DP mảng | `dp[i] = dp[i-1] + 1` nếu hiệu giữ nguyên | `O(n)` | `O(n)` | dễ trình bày, dễ debug |
| 4. Một biến đếm | thu gọn DP | **`O(n)`** | **`O(1)`** | **nộp bài**, phỏng vấn |
| 5. Công thức theo dải | `m * (m - 1) / 2` cho mỗi dải hiệu bằng nhau | `O(n)` | `O(n)` | góc nhìn tổ hợp, kiểm chứng chéo |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `[b - a for a, b in zip([2, 4, 7], [4, 7])]`; (b) `Solution().numberOfArithmeticSlices([5, 5, 5])`; (c) `Solution().numberOfArithmeticSlices([1, 2, 4, 7])`; (d) `3 * 2 // 2`.

**Bài 2 (tính tay).** Với `nums = [1, 3, 5, 7, 9, 9, 9, 12]`, viết ra `diff`, các dải hiệu bằng nhau, và đếm số cấp số cộng bằng công thức `m * (m - 1) / 2` cho từng dải.

**Bài 3 (điền TODO).** Hoàn thành `count_todo(nums)` theo Cách 4 (biến đếm).

In [31]:
def count_todo(nums):
    # TODO 1: ans = 0, cur = 0
    # TODO 2: với i từ 2 đến len(nums) - 1: nếu hiệu giữ nguyên thì cur += 1, ngược lại cur = 0
    # TODO 3: ans += cur sau mỗi bước
    # TODO 4: return ans
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `count_dp2(nums)` theo Cách 3 (mảng `dp`) **không nhìn lại** mục 7. Kiểm tra với `[1,2,3,4]`, `[1]`, `[7,7,7,7]`.

**Bài 5 (biến thể / mini challenge).** Viết `count_at_least(nums, L)`: đếm số đoạn con liên tiếp là cấp số cộng có **độ dài `>= L`** (với `L >= 3`). Hãy tự rút công thức cho từng dải rồi kiểm chứng bằng vét cạn trước khi tin. Với `L = 3` phải trùng đáp án gốc.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[2, 3]`; `1`; `0` (hiệu `1, 2, 3` không bằng nhau); `3`. Bài 3: xem `count_counter` ở mục 7.
- **Mức 2 (vừa):** Bài 2: `diff = [2, 2, 2, 2, 0, 0, 3]`. Dải hiệu `2` dài `m = 4` ⇒ `4 * 3 / 2 = 6`; dải hiệu `0` dài `m = 2` ⇒ `1`; dải hiệu `3` dài `1` ⇒ `0`. Tổng `7`. Bài 4: `dp[i] = dp[i-1] + 1` khi hiệu giữ nguyên, `sum(dp)`.
- **Mức 3 (sâu):** Bài 5: trong đoạn cực đại `m` hiệu (tức `m + 1` phần tử), số đoạn con có độ dài `>= L` là `sum(m + 1 - len + 1 for len in range(L, m + 2))`; viết bằng vòng lặp trước rồi mới rút gọn công thức. Ở đây hai đoạn con có độ dài `>= 3` nằm hoàn toàn trong một dải nên không đếm trùng.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert [b - a for a, b in zip([2, 4, 7], [4, 7])] == [2, 3]
assert Solution().numberOfArithmeticSlices([5, 5, 5]) == 1
assert Solution().numberOfArithmeticSlices([1, 2, 4, 7]) == 0
assert 3 * 2 // 2 == 3

# Bài 2
nums2 = [1, 3, 5, 7, 9, 9, 9, 12]
diff2 = [b - a for a, b in zip(nums2, nums2[1:])]
assert diff2 == [2, 2, 2, 2, 0, 0, 3]
runs = [(v, len(list(g))) for v, g in groupby(diff2)]
assert runs == [(2, 4), (0, 2), (3, 1)]
assert sum(m * (m - 1) // 2 for _, m in runs) == 7 == Solution().numberOfArithmeticSlices(nums2) == count_brute(nums2)


# Bài 3
def count_todo_ans(nums):
    ans = cur = 0
    for i in range(2, len(nums)):
        if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
            cur += 1
        else:
            cur = 0
        ans += cur
    return ans


for nums, expected in examples:
    assert count_todo_ans(nums) == expected


# Bài 4
def count_dp2(nums):
    n = len(nums)
    dp = [0] * n
    for i in range(2, n):
        if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
            dp[i] = dp[i - 1] + 1
    return sum(dp)


assert [count_dp2(x) for x in ([1, 2, 3, 4], [1], [7, 7, 7, 7])] == [3, 0, 3]


# Bài 5
def count_at_least(nums, L):
    diff = [b - a for a, b in zip(nums, nums[1:])]
    ans = 0
    for _, grp in groupby(diff):
        m = len(list(grp))                                   # dải có m + 1 phần tử
        for length in range(L, m + 2):
            ans += (m + 1) - length + 1
    return ans


def count_at_least_brute(nums, L):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + L - 1, n) if is_arithmetic(nums[i:j + 1]))


random.seed(4132)
for _ in range(300):
    arr = [random.randint(-2, 2) for _ in range(random.randint(1, 14))]
    for L in (3, 4, 5):
        assert count_at_least(arr, L) == count_at_least_brute(arr, L), (arr, L)
assert count_at_least([1, 2, 3, 4], 3) == 3 and count_at_least([1, 2, 3, 4], 4) == 1
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (đếm theo điểm kết thúc, một biến)

```python
class Solution:
    def numberOfArithmeticSlices(self, nums: List[int]) -> int:
        ans = cur = 0
        for i in range(2, len(nums)):
            if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
                cur += 1
            else:
                cur = 0
            ans += cur
        return ans
```

**Checklist**

- [ ] Hiểu rõ **subarray = liên tiếp**, độ dài **`>= 3`**?
- [ ] Vòng lặp bắt đầu từ `i = 2` (cần 3 phần tử)? `n < 3` trả về `0` tự nhiên?
- [ ] Nhánh `else` reset `cur = 0`?
- [ ] `ans += cur` ở **mỗi** bước (không chỉ khi hiệu bằng nhau thì `+1`)?
- [ ] Đã thử `[1]`, `[1,2]`, `[7,7,7,7]`, dãy tăng-giảm xen kẽ, mảng `n = 5000` toàn số bằng nhau?
- [ ] Đã đối chiếu với vét cạn theo định nghĩa trên mảng ngắn?